# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"FAIR2 schema @id: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

`mlcroissant` allows us to inspect the different RecordSets available in the dataset, along with their fields (columns). Every entity (record set, field, column, etc.) is referred to by its `@id`.

In [ ]:
# List all record sets with their @id and field @ids
record_sets = dataset.metadata.record_sets

if not record_sets:
    print('No explicit record sets found in the "recordSet" field. Attempting to resolve from distribution object...')
    # Often, the dataset may have only one default record set, usually under the main file object/distribution
    # Let's examine them
    distributions = getattr(metadata, 'distributions', None)
    if not distributions:
        distributions = getattr(metadata, 'distribution', None)  # some schemas use 'distribution'
    if distributions:
        print("Distributions found (may serve as record sets):")
        for d in distributions:
            print(f"  @id: {getattr(d, 'id', None)}, Name: {getattr(d, 'name', None)}, Encoding: {getattr(d, 'encoding_format', None)}")
        # For mlcroissant, the 'record_set' arg sometimes accepts the distribution @id as well
    else:
        print('No record set or distribution information found. Please check the schema.')
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rec in record_sets:
        print(f"RecordSet: @id: {rec.id}, name: {getattr(rec, 'name', '-')}")
        # List fields in record set
        fields = getattr(rec, 'fields', getattr(rec, 'field', []))
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    @id: {f.id}, name: {getattr(f, 'name', getattr(f, 'label', None))}, dataType: {getattr(f, 'data_type', None)}")
        print("")
        # Optionally list columns
        columns = getattr(rec, 'columns', getattr(rec, 'column', []))
        if columns:
            print("  Columns:")
            for c in columns:
                print(f"    @id: {c.id}, name: {getattr(c, 'name', getattr(c, 'label', None))}, dataType: {getattr(c, 'data_type', None)}")
        print("")

# Print a sample record from what we deduce as main record set (if present)
print("\nSample record:")
# Good practice: try the main distribution record set @id
try_record_set = None
if not record_sets and distributions:
    try_record_set = getattr(distributions[0], 'id', None)
elif record_sets:
    try_record_set = record_sets[0].id
if try_record_set:
    for i, record in enumerate(dataset.records(record_set=try_record_set)):
        print(record)
        if i > 1:
            break
else:
    print("No accessible record set for sampling records.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We extract available record set `@id`s (either from `recordSet` or via the main distribution object) and load several rows into individual DataFrames.

In [ ]:
# Prepare list of record set @ids to extract
record_set_ids = []
record_sets = dataset.metadata.record_sets
if record_sets:
    record_set_ids = [r.id for r in record_sets]
else:
    # Attempt fallback to distribution object @id
    distributions = getattr(metadata, 'distributions', None)
    if not distributions:
        distributions = getattr(metadata, 'distribution', None)
    if distributions:
        record_set_ids = [getattr(d, 'id', None) for d in distributions if getattr(d, 'id', None)]

print(f"Record set @ids: {record_set_ids}")

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    # Sometimes all records are empty if the distribution isn't a records table
    if not df.empty:
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records from {rec_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records extracted for record set @id {rec_id}")

# Choose the first non-empty record set for further analysis
main_record_set_id = None
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Selected record set for further analysis: {main_record_set_id}")
else:
    print("No record set with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

We will select a representative numeric field by `@id` and a possible group/categorical field (if present) from the DataFrame.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Available columns in record set {main_record_set_id}:")
    print(df.columns.tolist())

    # Attempt to select a numeric field for filtering and normalization
    # Try common protein abundance/properties names
    numeric_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['abundance', 'mw', 'molecular_weight', 'coverage', 'count', 'peptide', 'intensity', 'pi'])]
    numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None
    print(f"Numeric field candidate: {numeric_field}")

    # Pick a grouping field (usually categorical)
    group_field_candidates = [c for c in df.columns if any(s in c.lower() for s in ['condition', 'group', 'compartment', 'sample', 'stimulus', 'type'])]
    group_field = group_field_candidates[0] if group_field_candidates else None

    # EDA: filter, normalize, and group
    if numeric_field:
        # Convert numeric_field to float
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nRecords with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)}")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group if possible
        if group_field and group_field in filtered_df:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No non-empty main DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot a histogram of the selected numeric field, or a boxplot by group if the field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if group_field and group_field in df:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:

- Load and inspect detailed Croissant metadata using `mlcroissant`
- Identify record set, field, and column `@id`s
- Extract tabular records into Pandas DataFrames
- Perform common EDA operations (filtering, normalization, grouping)
- Visualize feature distributions and group differences

This workflow provides a reproducible path for further mass spectrometry and proteomics data analysis. For richer downstream analysis, consult the schema for field semantics and reference the `@id` mapping for all data manipulations.